# 🤖 DOBOT CR10A - BƯỚC 2: HUẤN LUYỆN MÔ HÌNH THỊ GIÁC AI (YOLO11-SEG & YOLO11-POSE)
**Dự án**: Tự động sinh chương trình phun hiệu ứng mài bạc trên quần Jeans (Camera 3D Zivid + Robot Dobot CR10A)  
**Nền tảng**: Kaggle GPU (T4 x2 hoặc P100) / Google Colab / Local Ubuntu

---
### 📌 Hướng dẫn chạy nhanh trên Kaggle:
1. Ở cột bên phải mục **Input**, nhấn **`+ Add Input`** $\rightarrow$ **`Upload`** $\rightarrow$ chọn file **`kaggle_dataset_dobot_cr10a.zip`**.
2. Ở mục **Session Options (Accelerator)**, chọn **`GPU T4 x2`** hoặc **`GPU P100`**.
3. Nhấn nút **`Run All`** trên thanh công cụ và đợi khoảng **10 - 15 phút**.
4. Khi chạy xong Cell cuối cùng, tải file **`/kaggle/working/dobot_cr10a_trained_models.zip`** về máy tính để dùng cho **Bước 3 (Sinh quỹ đạo Robot 3D)**.

In [1]:
# ==============================================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN ULTRALYTICS & KIỂM TRA PHẦN CỨNG GPU
# ==============================================================================
!pip install -q -U ultralytics

import os
import torch
import ultralytics

ultralytics.checks()
print("=" * 65)
print(f"PyTorch version     : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device          : {torch.cuda.get_device_name(0)}")
print("=" * 65)

ERROR: Operation cancelled by user
^C
Traceback (most recent call last):
  File "/usr/bin/pip", line 8, in <module>
    sys.exit(main())
  File "/usr/lib/python3/dist-packages/pip/_internal/cli/main.py", line 70, in main
    return command.main(cmd_args)
  File "/usr/lib/python3/dist-packages/pip/_internal/cli/base_command.py", line 98, in main
    with self.main_context():
  File "/usr/lib/python3.10/contextlib.py", line 142, in __exit__
    next(self.gen)
  File "/usr/lib/python3/dist-packages/pip/_internal/cli/command_context.py", line 19, in main_context
    with self._main_context:
  File "/usr/lib/python3.10/contextlib.py", line 576, in __exit__
    raise exc_details[1]
  File "/usr/lib/python3.10/contextlib.py", line 153, in __exit__
    self.gen.throw(typ, value, traceback)
  File "/usr/lib/python3/dist-packages/pip/_internal/utils/temp_dir.py", line 70, in tempdir_registry
    yield _tempdir_registry
  File "/usr/lib/python3.10/contextlib.py", line 561, in __exit__
    if cb(*

ModuleNotFoundError: No module named 'torch'

In [ ]:
# ==============================================================================
# CELL 2: TỰ ĐỘNG DÒ TÌM DATASET & CẤU HÌNH THƯ MỤC LÀM VIỆC (CHỐNG LỖI READ-ONLY KAGGLE)
# ==============================================================================
# Lưu ý kỹ thuật trên Kaggle:
# - Thư mục /kaggle/input/ chỉ cho phép ĐỌC (Read-Only), trong khi Ultralytics
#   yêu cầu thư mục nhãn phải tên là 'labels/' và cần quyền GHI file .cache.
# - Đoạn code dưới đây tự động tìm dataset của bạn trong /kaggle/input/ và tạo
#   liên kết mềm (Symlink) sang /kaggle/working/ cho cả 2 bài toán SEG và POSE
#   mà không tốn dung lượng copy ảnh!

from pathlib import Path
import shutil
import yaml

def find_dataset_root():
    search_roots = [Path("/kaggle/input"), Path("/content"), Path("/home/nhan/RB3t_PJ/ai_src")]
    for root in search_roots:
        if not root.exists():
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            p = Path(dirpath)
            if (p / "images" / "train").exists() and (
                (p / "labels_seg" / "train").exists() or (p / "labels" / "train").exists()
            ):
                return p
    raise FileNotFoundError("Không tìm thấy dataset! Hãy chắc chắn bạn đã Add Input file kaggle_dataset_dobot_cr10a.zip trên Kaggle.")

SRC_DATASET = find_dataset_root()
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working_yolo")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print(f"✓ Đã tìm thấy Dataset gốc tại : {SRC_DATASET}")
print(f"✓ Thư mục làm việc (Writable) : {WORK_ROOT}")

def prepare_task_workspace(task_name, label_folder_name):
    task_dir = WORK_ROOT / "datasets" / f"jeans_{task_name}"
    if task_dir.exists():
        shutil.rmtree(task_dir)
    for split in ["train", "val"]:
        img_dst = task_dir / "images" / split
        lbl_dst = task_dir / "labels" / split
        img_dst.parent.mkdir(parents=True, exist_ok=True)
        lbl_dst.mkdir(parents=True, exist_ok=True)

        # Symlink thư mục ảnh để không tốn bộ nhớ
        os.symlink((SRC_DATASET / "images" / split).resolve(), img_dst)

        # Copy các file .txt nhãn sang vùng nhớ ghi được để YOLO tạo file .cache mượt mà
        src_lbl_dir = SRC_DATASET / label_folder_name / split
        for txt_file in src_lbl_dir.glob("*.txt"):
            shutil.copy2(txt_file, lbl_dst / txt_file.name)

    return task_dir

seg_label_src = "labels_seg" if (SRC_DATASET / "labels_seg").exists() else "labels"
SEG_DATASET_DIR = prepare_task_workspace("seg", seg_label_src)
POSE_DATASET_DIR = prepare_task_workspace("pose", "labels_pose")

# Xuất 2 file YAML chuẩn
SEG_YAML_PATH = WORK_ROOT / "data_seg.yaml"
POSE_YAML_PATH = WORK_ROOT / "data_pose.yaml"

seg_yaml_data = {
    "path": str(SEG_DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "garment"}
}
with open(SEG_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(seg_yaml_data, f, sort_keys=False)

pose_yaml_data = {
    "path": str(POSE_DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "kpt_shape": [12, 3],
    "flip_idx": [2, 1, 0, 3, 5, 4, 7, 6, 11, 10, 9, 8],
    "names": {0: "pants"}
}
with open(POSE_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(pose_yaml_data, f, sort_keys=False)

print(f"✓ Đã tạo cấu hình Segmentation : {SEG_YAML_PATH}")
print(f"✓ Đã tạo cấu hình Pose (12 kpt): {POSE_YAML_PATH}")

In [ ]:
# ==============================================================================
# CELL 3: HUẤN LUYỆN MÔ HÌNH 1 - YOLO11-SEG (PHÂN ĐOẠN VÙNG VẢI THÂN QUẦN)
# ==============================================================================
# Tham số tối ưu theo góp ý Hội đồng:
# - degrees=15.0: Xoay ngẫu nhiên +-15 độ mô phỏng đặt quần lệch góc trên bàn gá.
# - hsv_v=0.4, hsv_s=0.5: Tăng cường độ sáng/tương phản thích ứng vải Denim tối màu & sáng màu.

from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
SEG_MODEL_SIZE = "yolo11s-seg.pt"  # Có thể đổi thành yolo11n-seg.pt hoặc yolo11m-seg.pt

model_seg = YOLO(SEG_MODEL_SIZE)
results_seg = model_seg.train(
    data=str(SEG_YAML_PATH),
    epochs=100,
    patience=25,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project=str(WORK_ROOT / "runs_dobot"),
    name="yolo11s_seg_jeans",
    exist_ok=True,
    degrees=15.0,
    translate=0.1,
    scale=0.25,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    mosaic=0.5,
    plots=True
)
print("✓ Hoàn tất huấn luyện YOLO11-seg!")

In [ ]:
# ==============================================================================
# CELL 4: HUẤN LUYỆN MÔ HÌNH 2 - YOLO11-POSE (ĐỊNH VỊ 12 MỐC GIẢI PHẪU QUẦN)
# ==============================================================================
# Thứ tự 12 mốc giải phẫu:
# 0:waist_left, 1:waist_center, 2:waist_right, 3:crotch,
# 4:thigh_left, 5:thigh_right, 6:knee_left, 7:knee_right,
# 8:cuff_left_outer, 9:cuff_left_inner, 10:cuff_right_inner, 11:cuff_right_outer

POSE_MODEL_SIZE = "yolo11s-pose.pt"  # Có thể đổi thành yolo11n-pose.pt hoặc yolo11m-pose.pt

model_pose = YOLO(POSE_MODEL_SIZE)
results_pose = model_pose.train(
    data=str(POSE_YAML_PATH),
    epochs=120,
    patience=30,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project=str(WORK_ROOT / "runs_dobot"),
    name="yolo11s_pose_jeans",
    exist_ok=True,
    degrees=15.0,
    translate=0.1,
    scale=0.25,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    mosaic=0.5,
    pose=15.0,  # Tăng trọng số hàm mất mát Keypoint để định vị mốc giải phẫu siêu chính xác
    plots=True
)
print("✓ Hoàn tất huấn luyện YOLO11-pose!")

In [ ]:
# ==============================================================================
# CELL 5: ĐÁNH GIÁ CHỈ SỐ NGHIỆM THU (mAP50, mAP50-95, TỐC ĐỘ SUY LUẬN)
# ==============================================================================
best_seg_pt = WORK_ROOT / "runs_dobot" / "yolo11s_seg_jeans" / "weights" / "best.pt"
best_pose_pt = WORK_ROOT / "runs_dobot" / "yolo11s_pose_jeans" / "weights" / "best.pt"

val_seg = YOLO(str(best_seg_pt)).val(data=str(SEG_YAML_PATH), split="val")
val_pose = YOLO(str(best_pose_pt)).val(data=str(POSE_YAML_PATH), split="val")

print("\n" + "=" * 70)
print("📊 BẢNG TỔNG HỢP KẾT QUẢ NGHIỆM THU BƯỚC 2 (VALIDATION METRICS)")
print("=" * 70)
print(f"[1] MÔ HÌNH PHÂN ĐOẠN (YOLO11s-seg):")
print(f"    - Mask mAP50       : {val_seg.seg.map50 * 100:.2f} %")
print(f"    - Mask mAP50-95    : {val_seg.seg.map * 100:.2f} %  (Mục tiêu >= 90%)")
print(f"    - Tốc độ suy luận  : {val_seg.speed['inference']:.2f} ms/ảnh (Mục tiêu <= 30ms)")
print("-" * 70)
print(f"[2] MÔ HÌNH 12 MỐC GIẢI PHẪU (YOLO11s-pose):")
print(f"    - Pose OKS mAP50   : {val_pose.pose.map50 * 100:.2f} %")
print(f"    - Pose OKS mAP50-95: {val_pose.pose.map * 100:.2f} %  (Mục tiêu >= 90%)")
print(f"    - Tốc độ suy luận  : {val_pose.pose['inference'] if isinstance(val_pose.pose, dict) else val_pose.speed['inference']:.2f} ms/ảnh")
print("=" * 70)

In [ ]:
# ==============================================================================
# CELL 6: ĐÓNG GÓI TRỌNG SỐ BEST.PT & BIỂU ĐỒ BÁO CÁO ĐỂ TẢI VỀ MÁY (1-CLICK)
# ==============================================================================
import zipfile

export_dir = WORK_ROOT / "exported_models"
export_dir.mkdir(parents=True, exist_ok=True)

# Copy 2 file trọng số quan trọng nhất với tên rõ ràng
shutil.copy2(best_seg_pt, export_dir / "best_yolo11_seg.pt")
shutil.copy2(best_pose_pt, export_dir / "best_yolo11_pose.pt")

# Nén toàn bộ trọng số + biểu đồ huấn luyện (loss curves, PR curves, ảnh dự đoán) vào 1 file zip
zip_out = WORK_ROOT / "dobot_cr10a_trained_models.zip"
with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(export_dir / "best_yolo11_seg.pt", "weights/best_yolo11_seg.pt")
    zf.write(export_dir / "best_yolo11_pose.pt", "weights/best_yolo11_pose.pt")
    
    for run_name in ["yolo11s_seg_jeans", "yolo11s_pose_jeans"]:
        run_dir = WORK_ROOT / "runs_dobot" / run_name
        for f in run_dir.glob("*"):
            if f.is_file() and f.suffix.lower() in [".png", ".jpg", ".csv", ".yaml"]:
                zf.write(f, f"reports/{run_name}/{f.name}")

print("=" * 70)
print(f"🎉 ĐÃ ĐÓNG GÓI XONG KẾT QUẢ HUẤN LUYỆN!")
print(f"👉 Hãy tải file này ở mục Output bên phải Kaggle về máy:")
print(f"   {zip_out} ({zip_out.stat().st_size / (1024*1024):.2f} MB)")
print("=" * 70)